# SpectroNet-LSTM — Cardiac Anomaly Detection from Heart Sounds

Reproduction of **"SpectroNet-LSTM: An Interpretable Deep Learning Approach to
Cardiac Anomaly Detection Through Heart Sound Analysis"** (Sharma, Srivats,
P B, Mishra, Krithiga R, Nachiyappan S).

**On Kaggle:** add the dataset **"Heartbeat Sounds"** (search it in
*Add Data*; this is the public "Dangerous Heartbeat Dataset (DHD)" the paper
cites, sourced from the iStethoscope Pro app + clinical DigiScope trials).
It will mount at `/kaggle/input/heartbeat-sounds/`.

Pipeline (Section 3 of the paper):

1. Load `set_a.csv` / `set_b.csv` manifests (661 labeled clips, 5 classes:
   `normal`, `murmur`, `extrastole`, `artifact`, `extrahls`)
2. Preprocess audio: fixed-length normalization → band-pass filter →
   wavelet denoising
3. Convert to mel-spectrogram "heatmap" images
4. Extract + fuse features from 3 ImageNet backbones: **ResNet101, VGG16,
   InceptionV3**
5. Feed fused features into a **Conv1D + 2×LSTM** head (`SpectroNet-LSTM`)
6. Two-phase training: freeze backbones → fine-tune last 50 layers
7. Evaluate: accuracy / precision / recall / F1, confusion matrix, ROC-AUC
   — compare against baselines (SimpleRNN, GRU, TCN, MLP, CNN-LSTM)
8. Explainability: **SHAP** (global) + **LIME** (local) heatmaps per class

> This notebook is the *research* artifact. The equivalent *production*
> code (typed configs, unit-tested modules, CLI, Docker, CI) lives in
> `src/spectronet/` of the companion repo — see the project README.


## 0. Setup

In [ ]:
!pip install -q pywavelets librosa soundfile shap lime scikit-image ruff 2>/dev/null

import os, glob, math, random, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa, librosa.display
import pywt
from scipy.signal import butter, sosfiltfilt

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_curve, roc_auc_score
)

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

CLASSES = ["normal", "murmur", "extrastole", "artifact", "extrahls"]
NUM_CLASSES = len(CLASSES)

# Kaggle dataset root — adjust the folder name to whatever "Add Data" mounts
DATA_ROOT = Path("/kaggle/input/heartbeat-sounds")
print("Dataset present:", DATA_ROOT.exists())


## 1. Load the DHD manifest

`set_a.csv` / `set_b.csv` map filenames to one of the 5 classes. Rows with
missing labels (unlabeled clips used only for the segmentation sub-task) are
dropped, matching the paper's multiclass classification framing.


In [ ]:
def load_manifest(root: Path) -> pd.DataFrame:
    frames = []
    for set_name in ("set_a", "set_b"):
        csv_path = root / f"{set_name}.csv"
        if not csv_path.exists():
            continue
        df = pd.read_csv(csv_path)
        df = df.rename(columns={c: c.strip().lower() for c in df.columns})
        df = df.dropna(subset=["label"])
        df["label"] = df["label"].str.strip().str.lower()
        df = df[df["label"].isin(CLASSES)]

        def resolve(fname):
            fname = str(fname)
            cand = root / fname
            if cand.exists():
                return str(cand)
            return str(root / set_name / Path(fname).name)

        df["path"] = df["fname"].apply(resolve)
        df["source_set"] = set_name
        frames.append(df[["path", "label", "source_set"]])
    manifest = pd.concat(frames, ignore_index=True)
    manifest = manifest[manifest["path"].apply(lambda p: Path(p).exists())].reset_index(drop=True)
    return manifest

manifest = load_manifest(DATA_ROOT)
print(f"Total labeled clips: {len(manifest)}")
manifest["label"].value_counts()


In [ ]:
# Fig. 1 reproduction: class distribution pie chart
dist = manifest["label"].value_counts(normalize=True).reindex(CLASSES)
plt.figure(figsize=(5, 5))
plt.pie(dist.values, labels=dist.index, autopct="%1.0f%%", startangle=90)
plt.title("Distribution of dataset among classes")
plt.show()


## 2. Stratified 72 / 8 / 20 split

Per Section 3: 80% of the data is used for train+val (72% train / 8% val of
the total), 20% held out for test — split per-class to preserve the
distribution shown above.


In [ ]:
def stratified_split(manifest: pd.DataFrame, train=0.72, val=0.08, test=0.20, seed=SEED):
    rng = np.random.RandomState(seed)
    train_frames, val_frames, test_frames = [], [], []
    for label, group in manifest.groupby("label"):
        idx = group.sample(frac=1.0, random_state=rng.randint(0, 1_000_000)).index
        n = len(idx)
        n_test = int(round(n * test))
        n_val = int(round(n * val))
        test_frames.append(group.loc[idx[:n_test]])
        val_frames.append(group.loc[idx[n_test:n_test+n_val]])
        train_frames.append(group.loc[idx[n_test+n_val:]])
    train_df = pd.concat(train_frames).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    val_df = pd.concat(val_frames).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    test_df = pd.concat(test_frames).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return train_df, val_df, test_df

train_df, val_df, test_df = stratified_split(manifest)
print(f"train={len(train_df)}  val={len(val_df)}  test={len(test_df)}")


## 3. Audio preprocessing

Fixed-length normalization → band-pass filter (20-400 Hz, the heart-sound
band) → wavelet denoising (`db4`, soft-threshold), matching the paper's
Section 1 description of the cleaning pipeline.


In [ ]:
SAMPLE_RATE = 4000
CLIP_SECONDS = 5.0
BANDPASS_LOW, BANDPASS_HIGH = 20.0, 400.0
WAVELET, WAVELET_LEVEL = "db4", 4

def fix_length(signal, sr, seconds):
    target = int(sr * seconds)
    if len(signal) >= target:
        return signal[:target]
    return np.pad(signal, (0, target - len(signal)))

def bandpass_filter(signal, sr, low, high, order=4):
    nyq = 0.5 * sr
    sos = butter(order, [max(low/nyq, 1e-4), min(high/nyq, 0.999)], btype="band", output="sos")
    return sosfiltfilt(sos, signal)

def wavelet_denoise(signal, wavelet=WAVELET, level=WAVELET_LEVEL):
    coeffs = pywt.wavedec(signal, wavelet, level=level)
    sigma = np.median(np.abs(coeffs[-1])) / 0.6745 if len(coeffs[-1]) else 0.0
    thresh = sigma * np.sqrt(2 * np.log(max(len(signal), 2)))
    denoised = [coeffs[0]] + [pywt.threshold(c, thresh, mode="soft") for c in coeffs[1:]]
    rec = pywt.waverec(denoised, wavelet)
    return rec[:len(signal)]

def preprocess_signal(signal):
    signal = np.asarray(signal, dtype=np.float32)
    if signal.size == 0:
        signal = np.zeros(int(SAMPLE_RATE * CLIP_SECONDS), dtype=np.float32)
    signal = fix_length(signal, SAMPLE_RATE, CLIP_SECONDS)
    signal = bandpass_filter(signal, SAMPLE_RATE, BANDPASS_LOW, BANDPASS_HIGH)
    signal = wavelet_denoise(signal)
    peak = np.max(np.abs(signal)) or 1.0
    return (signal / peak).astype(np.float32)

def load_and_clean(path):
    signal, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True)
    return preprocess_signal(signal)


## 4. Spectrograms, waveforms, MFCCs per class (Fig. 3-22 reproduction)

Visual sanity check mirroring the paper's per-class spectrum/waveform/
spectrogram figures (normal shows tight low-frequency energy and periodic
lub-dub; murmur/artifact/extrahls show broader or irregular energy spread).


In [ ]:
N_FFT, HOP_LENGTH, N_MELS, N_MFCC = 1024, 256, 128, 40
IMAGE_SIZE = 224

def mel_spectrogram(signal):
    mel = librosa.feature.melspectrogram(y=signal, sr=SAMPLE_RATE, n_fft=N_FFT,
                                          hop_length=HOP_LENGTH, n_mels=N_MELS)
    return librosa.power_to_db(mel, ref=np.max)

fig, axes = plt.subplots(len(CLASSES), 3, figsize=(14, 3 * len(CLASSES)))
for i, cls in enumerate(CLASSES):
    row = manifest[manifest["label"] == cls].iloc[0]
    sig = load_and_clean(row["path"])
    mfcc = librosa.feature.mfcc(y=sig, sr=SAMPLE_RATE, n_mfcc=N_MFCC)
    mel = mel_spectrogram(sig)

    axes[i, 0].plot(sig); axes[i, 0].set_title(f"{cls}: waveform")
    axes[i, 1].imshow(mfcc, aspect="auto", origin="lower", cmap="coolwarm")
    axes[i, 1].set_title(f"{cls}: MFCCs")
    axes[i, 2].imshow(mel, aspect="auto", origin="lower", cmap="magma")
    axes[i, 2].set_title(f"{cls}: mel-spectrogram")
plt.tight_layout()
plt.show()


## 5. Spectrogram → RGB image for the CNN backbones

In [ ]:
def spectrogram_to_rgb(log_mel, image_size=IMAGE_SIZE):
    normed = (log_mel - log_mel.min()) / (np.ptp(log_mel) + 1e-8)
    normed = (normed * 255.0).astype(np.uint8)
    f_idx = np.linspace(0, normed.shape[0]-1, image_size).astype(int)
    t_idx = np.linspace(0, normed.shape[1]-1, image_size).astype(int)
    resized = normed[f_idx][:, t_idx]
    return np.stack([resized]*3, axis=-1)

def path_to_image(path):
    sig = load_and_clean(path)
    log_mel = mel_spectrogram(sig)
    return spectrogram_to_rgb(log_mel).astype(np.float32)

def one_hot(label):
    vec = np.zeros(NUM_CLASSES, dtype=np.float32)
    vec[CLASSES.index(label)] = 1.0
    return vec


## 6. tf.data pipelines

Streams `(image, one_hot_label)` pairs so the full 661-clip dataset never
needs to be held in memory at once.


In [ ]:
BATCH_SIZE = 32

def make_dataset(df, shuffle=False):
    def gen():
        rows = df.sample(frac=1.0).itertuples() if shuffle else df.itertuples()
        for row in rows:
            yield path_to_image(row.path), one_hot(row.label)
    sig = (
        tf.TensorSpec(shape=(IMAGE_SIZE, IMAGE_SIZE, 3), dtype=tf.float32),
        tf.TensorSpec(shape=(NUM_CLASSES,), dtype=tf.float32),
    )
    ds = tf.data.Dataset.from_generator(gen, output_signature=sig)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(train_df, shuffle=True)
val_ds = make_dataset(val_df)
test_ds = make_dataset(test_df)


## 7. Model architecture (Fig. 2)

Three ImageNet-pretrained backbones (ResNet101, VGG16, InceptionV3) each
pool their own spectrogram embedding; embeddings are concatenated
("feature fusion") and fed into a `RepeatVector -> Conv1D -> 2xLSTM ->
Dense -> Softmax` head — the paper's fine-tuned LSTM.


In [ ]:
BACKBONES = {
    "resnet101": (tf.keras.applications.ResNet101, tf.keras.applications.resnet.preprocess_input, 224),
    "vgg16": (tf.keras.applications.VGG16, tf.keras.applications.vgg16.preprocess_input, 224),
    "inception_v3": (tf.keras.applications.InceptionV3, tf.keras.applications.inception_v3.preprocess_input, 299),
}

def build_backbone(name, trainable=False):
    factory, _, size = BACKBONES[name]
    base = factory(include_top=False, weights="imagenet", pooling="avg", input_shape=(size, size, 3))
    base.trainable = trainable
    base._name = f"{name}_backbone"
    return base

def preprocess_for(name, images):
    _, preprocess_fn, size = BACKBONES[name]
    resized = tf.image.resize(images, (size, size))
    return preprocess_fn(resized)

def unfreeze_last_n(backbone, n):
    backbone.trainable = True
    freeze_until = max(0, len(backbone.layers) - n)
    for layer in backbone.layers[:freeze_until]:
        layer.trainable = False

def build_fusion_model():
    image_input = layers.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3), name="spectrogram")
    backbones = {name: build_backbone(name) for name in BACKBONES}
    fused = layers.Concatenate(name="fused_features")(
        [backbones[name](preprocess_for(name, image_input)) for name in BACKBONES]
    )

    x = layers.RepeatVector(8)(fused)
    x = layers.Conv1D(128, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling1D(2, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.LSTM(128, return_sequences=True)(x)
    x = layers.LSTM(64)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

    model = models.Model(image_input, outputs, name="spectronet_lstm")
    model._backbones = backbones
    return model

model = build_fusion_model()
model.summary()


## 8. Two-phase training

**Phase 1** — freeze all backbone layers, train only the LSTM head
(Adam, lr=1e-4, categorical cross-entropy, early stopping patience=15).
**Phase 2 (fine-tuning)** — unlock the last 50 layers of each backbone
and continue training at a reduced learning rate.


In [ ]:
model.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
              loss="categorical_crossentropy", metrics=["accuracy"])

cb = [
    callbacks.EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True),
    callbacks.ModelCheckpoint("spectronet_lstm_head.keras", monitor="val_loss", save_best_only=True),
]

history_head = model.fit(train_ds, validation_data=val_ds, epochs=20, callbacks=cb, verbose=2)


In [ ]:
for backbone in model._backbones.values():
    unfreeze_last_n(backbone, n=50)

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss="categorical_crossentropy", metrics=["accuracy"])

cb_ft = [
    callbacks.EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True),
    callbacks.ModelCheckpoint("spectronet_lstm_finetuned.keras", monitor="val_loss", save_best_only=True),
]

history_ft = model.fit(train_ds, validation_data=val_ds, epochs=20, callbacks=cb_ft, verbose=2)


In [ ]:
# Fig. 23 / Fig. 24 reproduction: accuracy & loss before/after fine-tuning
def plot_history(history, title):
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    ax[0].plot(history.history["accuracy"], label="train")
    ax[0].plot(history.history["val_accuracy"], label="val")
    ax[0].set_title(f"Model accuracy — {title}"); ax[0].legend()
    ax[1].plot(history.history["loss"], label="train")
    ax[1].plot(history.history["val_loss"], label="val")
    ax[1].set_title(f"Model loss — {title}"); ax[1].legend()
    plt.tight_layout(); plt.show()

plot_history(history_head, "before fine-tuning")
plot_history(history_ft, "after fine-tuning (last 50 layers)")


## 9. Evaluation — Table 1, confusion matrix, ROC (Fig. 25-27)


In [ ]:
def evaluate(model, ds):
    y_true, y_pred = [], []
    for images, labels in ds:
        y_true.append(labels.numpy())
        y_pred.append(model.predict(images, verbose=0))
    return np.concatenate(y_true), np.concatenate(y_pred)

y_true, y_pred_proba = evaluate(model, test_ds)
y_true_idx, y_pred_idx = y_true.argmax(1), y_pred_proba.argmax(1)

results = {
    "Fine-Tuned LSTM": {
        "Accuracy": accuracy_score(y_true_idx, y_pred_idx),
        "Precision": precision_score(y_true_idx, y_pred_idx, average="weighted", zero_division=0),
        "Recall": recall_score(y_true_idx, y_pred_idx, average="weighted", zero_division=0),
        "F1 Score": f1_score(y_true_idx, y_pred_idx, average="weighted", zero_division=0),
    }
}
pd.DataFrame(results).T.round(2)


In [ ]:
cm = confusion_matrix(y_true_idx, y_pred_idx, labels=range(NUM_CLASSES))
cm_norm = cm.astype(float) / np.clip(cm.sum(axis=1, keepdims=True), 1, None)

plt.figure(figsize=(6, 5))
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="RdPu", xticklabels=CLASSES, yticklabels=CLASSES)
plt.xlabel("Prediction"); plt.ylabel("Truth"); plt.title("Normalized Confusion Matrix")
plt.show()


In [ ]:
plt.figure(figsize=(6, 6))
for i, cls in enumerate(["normal", "extrastole", "murmur"]):
    idx = CLASSES.index(cls)
    fpr, tpr, _ = roc_curve(y_true[:, idx], y_pred_proba[:, idx])
    auc = roc_auc_score(y_true[:, idx], y_pred_proba[:, idx])
    plt.plot(fpr, tpr, label=f"{cls} (AUC = {auc:.2f})")
plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate"); plt.title("ROC Curve")
plt.legend(); plt.show()


## 10. Baselines — Table 1 comparison

SimpleRNN, GRU, TCN, MLP, and a plain CNN-LSTM ("SpectroNet-LSTM" in the
paper's own baseline row) trained directly on MFCC sequences, for the
radar-plot comparison against the fine-tuned fusion model.


In [ ]:
SEQ_LEN = 130  # ~ time frames for a 5s clip at hop_length=256, sr=4000

def path_to_mfcc(path):
    sig = load_and_clean(path)
    mfcc = librosa.feature.mfcc(y=sig, sr=SAMPLE_RATE, n_mfcc=N_MFCC,
                                 n_fft=N_FFT, hop_length=HOP_LENGTH).T  # (T, n_mfcc)
    if mfcc.shape[0] >= SEQ_LEN:
        return mfcc[:SEQ_LEN]
    pad = np.zeros((SEQ_LEN - mfcc.shape[0], N_MFCC), dtype=np.float32)
    return np.vstack([mfcc, pad]).astype(np.float32)

def make_mfcc_dataset(df, shuffle=False):
    def gen():
        rows = df.sample(frac=1.0).itertuples() if shuffle else df.itertuples()
        for row in rows:
            yield path_to_mfcc(row.path), one_hot(row.label)
    sig = (tf.TensorSpec(shape=(SEQ_LEN, N_MFCC), dtype=tf.float32),
           tf.TensorSpec(shape=(NUM_CLASSES,), dtype=tf.float32))
    return tf.data.Dataset.from_generator(gen, output_signature=sig).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

mfcc_train, mfcc_val, mfcc_test = (make_mfcc_dataset(train_df, True),
                                    make_mfcc_dataset(val_df),
                                    make_mfcc_dataset(test_df))

def build_simple_rnn():
    i = layers.Input((SEQ_LEN, N_MFCC)); x = layers.SimpleRNN(64, return_sequences=True)(i)
    x = layers.SimpleRNN(32)(x); x = layers.Dense(64, activation="relu")(x)
    return models.Model(i, layers.Dense(NUM_CLASSES, activation="softmax")(x), name="simple_rnn")

def build_gru():
    i = layers.Input((SEQ_LEN, N_MFCC)); x = layers.GRU(64, return_sequences=True)(i)
    x = layers.GRU(32)(x); x = layers.Dense(64, activation="relu")(x)
    return models.Model(i, layers.Dense(NUM_CLASSES, activation="softmax")(x), name="gru")

def build_tcn():
    i = layers.Input((SEQ_LEN, N_MFCC)); x = i
    for d in (1, 2, 4, 8):
        x = layers.Conv1D(64, 3, padding="causal", dilation_rate=d, activation="relu")(x)
    x = layers.GlobalAveragePooling1D()(x); x = layers.Dense(64, activation="relu")(x)
    return models.Model(i, layers.Dense(NUM_CLASSES, activation="softmax")(x), name="tcn")

def build_mlp():
    i = layers.Input((SEQ_LEN, N_MFCC)); x = layers.Flatten()(i)
    x = layers.Dense(256, activation="relu")(x); x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation="relu")(x)
    return models.Model(i, layers.Dense(NUM_CLASSES, activation="softmax")(x), name="mlp")

def build_cnn_lstm():
    i = layers.Input((SEQ_LEN, N_MFCC)); x = layers.Conv1D(64, 3, padding="same", activation="relu")(i)
    x = layers.MaxPooling1D(2)(x); x = layers.LSTM(64, return_sequences=True)(x)
    x = layers.LSTM(32)(x); x = layers.Dense(64, activation="relu")(x)
    return models.Model(i, layers.Dense(NUM_CLASSES, activation="softmax")(x), name="cnn_lstm")

BASELINE_BUILDERS = {"SimpleRNN": build_simple_rnn, "GRU": build_gru, "TCN": build_tcn,
                      "MLP": build_mlp, "CNN-LSTM": build_cnn_lstm}


In [ ]:
baseline_results = dict(results)  # seed with the fine-tuned fusion model's result
for name, builder in BASELINE_BUILDERS.items():
    print(f"Training baseline: {name}")
    m = builder()
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss="categorical_crossentropy", metrics=["accuracy"])
    m.fit(mfcc_train, validation_data=mfcc_val, epochs=20,
          callbacks=[callbacks.EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True)],
          verbose=0)
    yt, yp = evaluate(m, mfcc_test)
    yt_idx, yp_idx = yt.argmax(1), yp.argmax(1)
    baseline_results[name] = {
        "Accuracy": accuracy_score(yt_idx, yp_idx),
        "Precision": precision_score(yt_idx, yp_idx, average="weighted", zero_division=0),
        "Recall": recall_score(yt_idx, yp_idx, average="weighted", zero_division=0),
        "F1 Score": f1_score(yt_idx, yp_idx, average="weighted", zero_division=0),
    }

table1 = pd.DataFrame(baseline_results).T.round(2).sort_values("F1 Score", ascending=False)
table1


In [ ]:
# Radar plot (Fig. 26 reproduction)
metrics = ["Accuracy", "Precision", "Recall", "F1 Score"]
angles = np.linspace(0, 2*np.pi, len(metrics), endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
for model_name, row in table1.iterrows():
    values = row[metrics].tolist(); values += values[:1]
    ax.plot(angles, values, label=model_name)
    ax.fill(angles, values, alpha=0.05)
ax.set_xticks(angles[:-1]); ax.set_xticklabels(metrics)
ax.set_title("Model Performance Comparison")
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))
plt.show()


## 11. Explainable AI — SHAP + LIME (Fig. 30/31)

LIME gives a *local* explanation (which spectrogram regions drove *this*
prediction); SHAP gives a *global* per-feature attribution. Together they
let a clinician sanity-check the model against known acoustic cues (e.g.
murmur = broadband low/mid-frequency energy between S1/S2).


In [ ]:
from lime import lime_image
from skimage.segmentation import mark_boundaries

def predict_fn(images):
    return model.predict(images.astype(np.float32), verbose=0)

explainer = lime_image.LimeImageExplainer()

fig, axes = plt.subplots(len(CLASSES), 2, figsize=(8, 3 * len(CLASSES)))
for i, cls in enumerate(CLASSES):
    row = test_df[test_df["label"] == cls].iloc[0] if (test_df["label"] == cls).any() else manifest[manifest["label"] == cls].iloc[0]
    img = path_to_image(row.path)
    pred_idx = int(np.argmax(predict_fn(img[None])[0]))

    explanation = explainer.explain_instance(img.astype("double"), predict_fn,
                                              top_labels=NUM_CLASSES, hide_color=0, num_samples=300)
    temp, mask = explanation.get_image_and_mask(pred_idx, positive_only=True,
                                                 num_features=10, hide_rest=False)

    axes[i, 0].imshow(img.astype(np.uint8)); axes[i, 0].set_title(f"{cls}: original"); axes[i, 0].axis("off")
    axes[i, 1].imshow(mark_boundaries(temp / 255.0, mask)); axes[i, 1].set_title(f"{cls}: LIME"); axes[i, 1].axis("off")
plt.tight_layout()
plt.show()


In [ ]:
import shap

background = np.stack([path_to_image(p) for p in train_df["path"].sample(16, random_state=SEED)])
sample = np.stack([path_to_image(p) for p in test_df["path"].sample(3, random_state=SEED)])

shap_explainer = shap.GradientExplainer(model, background)
shap_values = shap_explainer.shap_values(sample)

shap.image_plot(shap_values, sample / 255.0)


## 12. Save artifacts

Persist the fine-tuned model + the Table-1 comparison so the production
pipeline (`src/spectronet`) or a downstream serving step can pick them up.


In [ ]:
model.save("spectronet_lstm_final.keras")
table1.to_csv("model_comparison_table.csv")
print("Saved: spectronet_lstm_final.keras, model_comparison_table.csv")


## Next steps (productionizing this notebook)

This notebook is intentionally research-first (inline functions, eager
plotting, one dataset). The companion repo turns the same logic into a
tested, deployable package:

| Notebook section | Production module |
|---|---|
| §1-2 manifest + split | `spectronet.data.dataset` |
| §3 preprocessing | `spectronet.data.preprocessing` |
| §4-6 spectrograms/tf.data | `spectronet.features.spectrogram` |
| §7 model | `spectronet.models.fusion_model`, `.lstm_head` |
| §8 training | `spectronet.training.train` |
| §9-10 evaluation | `spectronet.evaluation.metrics` |
| §11 XAI | `spectronet.explainability.xai` |

Run `python -m spectronet.pipeline --config configs/config.yaml` for the
end-to-end CLI equivalent, or see the repo `README.md` for Docker/CI usage.
